# Linux Disk Partitioning and Creating Filesystems (Educational Notebook)
This notebook builds on the installation-time overview of partitioning and filesystems from `week1/Lesson2-Linux_Installation_Partitioning_Filesystems_and_Swap.ipynb`, and focuses on the command-line tools themselves: inspecting disks, partitioning them with `fdisk`/`parted`/`gdisk`, and creating filesystems with `mkfs`.

## 1. Why Partition a Disk? (Recap)

A disk is divided into **partitions** so that different parts of the filesystem hierarchy - or different operating systems - can live in independently-manageable regions of the same physical disk. Partitioning happens once at (or after) installation time; this lesson covers the actual tools used to do it and to prepare the resulting partitions for use.


## 2. Viewing Existing Partitions and Disks

Before changing anything, always start by looking at what's already there.

```bash
lsblk                        # tree view of block devices and their partitions
lsblk -o NAME,SIZE,FSTYPE,MOUNTPOINT,TYPE   # a more detailed view
blkid                          # filesystem type and UUID of each partition
fdisk -l                         # list partition tables for all disks (requires root)
parted -l                          # list partition tables, including the table type (msdos/gpt)
cat /proc/partitions                 # a low-level kernel view of every partition
```

`lsblk` (list block devices) is usually the fastest way to get your bearings before reaching for an editing tool.


## 3. Partitioning with `fdisk`

`fdisk` is a classic, interactive, menu-driven partition editor. It historically only understood MBR partition tables, but modern versions also support GPT.

```bash
sudo fdisk /dev/sdb        # open the interactive editor for a disk (never point this at your boot disk casually)
```

Inside the `fdisk` prompt, common single-letter commands:

| Key | Action |
|---|---|
| `p` | Print the current partition table |
| `n` | Create a new partition |
| `d` | Delete a partition |
| `t` | Change a partition's type code |
| `l` | List known partition type codes |
| `w` | Write changes to disk and exit |
| `q` | Quit without saving |

Nothing is written to disk until you press `w` - you can freely experiment and press `q` to back out if something looks wrong.


## 4. Partitioning with `parted`

`parted` is a more modern partition editor that natively understands both MBR and GPT, and can be driven either interactively or scripted with a single command line - useful for automation.

```bash
sudo parted /dev/sdb                      # open the interactive editor
(parted) print                               # show the current partition table
(parted) mklabel gpt                           # initialize a new, empty GPT partition table (destroys existing data!)
(parted) mkpart primary ext4 0% 50%              # create a partition spanning the first half of the disk
(parted) resizepart 1 75%                          # resize partition 1 to end at 75% of the disk

# the same operations scripted, non-interactively:
sudo parted /dev/sdb mklabel gpt
sudo parted /dev/sdb mkpart primary ext4 0% 100%
```

Unlike `fdisk`, `parted`'s changes to the partition table generally take effect immediately as each command runs, not only at a final "write" step - so extra care is warranted.


## 5. Partitioning with `gdisk`

`gdisk` is `fdisk`'s GPT-specific counterpart, with a very similar interactive command set (`p`, `n`, `d`, `w`, `q` all behave the same way), but built specifically around GPT concepts like partition GUIDs and type codes.

```bash
sudo gdisk /dev/sdb
```

If you already know `fdisk`'s interactive commands, `gdisk` will feel immediately familiar - it exists mainly because early versions of `fdisk` had incomplete GPT support.


## 6. Creating Filesystems with `mkfs`

Once a partition exists, it needs to be **formatted** with a filesystem before Linux can store files on it. `mkfs` is a front-end that dispatches to the right filesystem-specific tool based on the type you specify.

```bash
sudo mkfs.ext4 /dev/sdb1                # format as ext4
sudo mkfs.xfs /dev/sdb1                   # format as xfs
sudo mkfs.vfat -F32 /dev/sdb1               # format as FAT32 (e.g. for an EFI System Partition)
sudo mkfs -t ext4 /dev/sdb1                   # equivalent generic form: mkfs -t <type>

sudo mkfs.ext4 -L data /dev/sdb1                # assign a filesystem label at creation time
sudo mkfs.ext4 -n /dev/sdb1                       # dry run: show what would happen without writing anything
```

Formatting a partition destroys any data that was previously on it - always confirm the device name (`lsblk`/`blkid`) before running `mkfs`.


## 7. Filesystem Labels and UUIDs

Every filesystem has a **UUID** (a unique identifier generated at creation time) and, optionally, a human-readable **label**. Both are useful for referring to a filesystem in a way that doesn't depend on which device name (`/dev/sdb1` and so on) the kernel happens to assign it this boot.

```bash
blkid /dev/sdb1                    # show a filesystem's UUID, label, and type
lsblk -o NAME,LABEL,UUID,FSTYPE      # a tabular view across all block devices

e2label /dev/sdb1 data                 # set/change the label on an ext2/3/4 filesystem after creation
xfs_admin -L data /dev/sdb1              # equivalent for xfs
```


## 8. Checking and Tuning Filesystems

```bash
sudo fsck /dev/sdb1              # check a filesystem for errors and attempt to repair them (unmount it first!)
sudo fsck -f /dev/sdb1             # force a check even if the filesystem appears clean
sudo e2fsck -y /dev/sdb1             # ext-family specific check, auto-answering "yes" to repair prompts

sudo tune2fs -l /dev/sdb1              # list ext-family filesystem parameters (creation time, mount count, etc.)
sudo tune2fs -L newlabel /dev/sdb1       # another way to change an ext filesystem's label
```

Never run `fsck` on a mounted, in-use filesystem - unmount it first (or check it from separate rescue media if it's the root filesystem), since repairing a filesystem the kernel is actively writing to can cause further corruption.


## Hands-on

If you have a spare disk or a virtual machine (never practice on a disk with data you care about):

```bash
lsblk
sudo parted /dev/sdb mklabel gpt
sudo parted /dev/sdb mkpart primary ext4 0% 100%
lsblk
sudo mkfs.ext4 -L scratch /dev/sdb1
blkid /dev/sdb1
sudo fsck -f /dev/sdb1
```


## Review Questions

1. What is the quickest command to get an overview of every disk and partition on a system?
2. Name one practical difference between how `fdisk` and `parted` apply changes (when do they actually hit the disk?).
3. Why does `gdisk` exist alongside `fdisk`?
4. What does `mkfs.ext4 -n /dev/sdb1` do, and why is it useful before running the real command?
5. Why might you prefer to reference a filesystem by its UUID rather than by `/dev/sdb1` in scripts or configuration?
6. What tool would you use to change the label of an already-created ext4 filesystem, without reformatting it?
7. Why must a filesystem be unmounted before running `fsck` on it?
8. What destructive step in `parted` initializes a brand-new, empty partition table, and what happens to any existing data when you run it?


# Cheat Sheet

```
Inspect:
  lsblk | lsblk -o NAME,SIZE,FSTYPE,MOUNTPOINT,TYPE
  blkid          fdisk -l          parted -l

Partition (MBR-focused, interactive):
  fdisk /dev/sdX      n (new)  d (delete)  t (type)  p (print)  w (write)  q (quit)

Partition (GPT-focused, interactive):
  gdisk /dev/sdX        same key commands as fdisk

Partition (scriptable, MBR or GPT):
  parted /dev/sdX mklabel gpt
  parted /dev/sdX mkpart primary ext4 0% 100%
  parted /dev/sdX resizepart 1 75%

Create filesystems:
  mkfs.ext4 /dev/sdX1     mkfs.xfs /dev/sdX1     mkfs.vfat -F32 /dev/sdX1
  mkfs -t <type> /dev/sdX1        mkfs.ext4 -L <label> /dev/sdX1

Labels/UUIDs:
  blkid /dev/sdX1        e2label /dev/sdX1 <label>       xfs_admin -L <label> /dev/sdX1

Check/tune:
  fsck /dev/sdX1 (unmounted only)      tune2fs -l /dev/sdX1
```
